# Traffic Demand Forecasting — Final Solution

**Competition:** Flipkart Grid 6.0 — Traffic Management & Travel Demand Forecast  
**Metric:** `max(0, 100 × R²)`  
**Final LB Score:** **91.50** (`submission_v9_bag.csv`)

---

## Approach Summary

The task is **regression** on a smooth spatiotemporal demand surface. The dataset covers:
- **Train:** day 48 (full 24h, tmin 0–1425) + day 49 (00:00–02:00, tmin 0–120)
- **Test:** day 49, 02:15–13:45 (tmin 135–825)

99.94% of test geohashes appear in the training data, so the problem reduces to interpolating a known spatial surface forward in time.

### What worked vs. what didn't

| Approach | OOF R² | LB Score | Issue |
|----------|--------|----------|-------|
| Geohash target encoding + Ridge stack | 0.953 | 89.0% | TE = all-day mean, biases daytime-only test |
| LGBM + `abs_time` feature | 0.950+ | 87.1% | test abs_times 69255–69945 all beyond training max 69240 → OOD |
| **LGBM + geohash categorical + day-48 lag, 3-seed bag** | 0.959 | **91.5% ✓** | No bias, no OOD |

---

## Feature Engineering

All features are deterministic and leakage-free (no target statistics):

| Feature | Description |
|---------|-------------|
| `lat`, `lon` | Decoded from geohash (base-32 → WGS-84 center) |
| `tmin` | `timestamp` converted to minutes-of-day (0–1425) |
| `hour` | Integer hour of day (`tmin // 60`) |
| `sin1/cos1`, `sin2/cos2`, `sin3/cos3` | Cyclic time harmonics at 1×, 2×, 3× daily frequency |
| `Temperature` | Median-imputed; `Temp_missing` flag added |
| `NumberofLanes` | Cast to float |
| `RoadType`, `LargeVehicles`, `Landmarks`, `Weather` | Native LGBM categoricals (NaN → `"Missing"`) |
| `gh6` | Full geohash string as native LGBM categorical (avoids all-day-mean bias of target encoding) |
| `d48_demand` | **Lag feature:** day-48 demand at the same (geohash, tmin). 88.9% test coverage; day-48 rows set to NaN to avoid self-reference |

**Key insight — why `tmin` not `abs_time`:**  
Absolute time for all test rows (69255–69945) falls entirely beyond the training maximum (69240). Using it as a feature means every test row gets clipped to the same bucket — useless. `tmin` (minutes within a day) keeps test values (135–825) inside the training range (0–1425) and generalises correctly.

**Key insight — why geohash categorical not target encoding:**  
Target encoding computes each geohash's *all-day average demand*. But test rows are only from 02:15–13:45, so the all-day mean is a biased proxy. Using the raw geohash as a native LGBM categorical lets the model learn the full demand profile per location without that bias.

---

## Model

**LightGBM** (`num_leaves=127`, `n_estimators=1000`, `learning_rate=0.03`) trained on all features above.  
**3-seed bagging** (seeds 42, 123, 2024) — predictions averaged over three independently seeded full-train fits.  
This reduces variance without introducing the bias that larger bags (5+) caused in earlier experiments.

**Tool used:** LightGBM 4.6.0, scikit-learn 1.x, pandas, numpy. Hardware: MacBook M1 Pro, `n_jobs=-1`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

# ── Load data ──
train = pd.read_csv("dataset/train.csv")
test  = pd.read_csv("dataset/test.csv")

# Convert "H:M" timestamp to minutes-of-day
for d in (train, test):
    d["tmin"] = d["timestamp"].map(
        lambda s: int(s.split(":")[0]) * 60 + int(s.split(":")[1])
    )

print("train:", train.shape, "| test:", test.shape)
print("Train tmin range:", train.tmin.min(), "–", train.tmin.max())
print("Test  tmin range:", test.tmin.min(),  "–", test.tmin.max())

In [ ]:
# ── Geohash decoder: base-32 string → (lat, lon) center ──
_B32 = "0123456789bcdefghjkmnpqrstuvwxyz"

def decode_gh(gh):
    a = [-90., 90.]; o = [-180., 180.]; e = True
    for c in gh:
        cd = _B32.index(c)
        for m in (16, 8, 4, 2, 1):
            if e:  mid = (o[0] + o[1]) / 2; o[0 if cd & m else 1] = mid
            else:  mid = (a[0] + a[1]) / 2; a[0 if cd & m else 1] = mid
            e = not e
    return (a[0] + a[1]) / 2, (o[0] + o[1]) / 2

GH2LL = {g: decode_gh(g) for g in pd.concat([train.geohash, test.geohash]).unique()}

CAT_COLS = ["RoadType", "LargeVehicles", "Landmarks", "Weather"]

def build_features(df):
    d = df.copy()
    d["lat"]  = d.geohash.map(lambda g: GH2LL[g][0])
    d["lon"]  = d.geohash.map(lambda g: GH2LL[g][1])
    d["hour"] = d.tmin // 60
    # Cyclic time-of-day harmonics (NO abs_time — it is OOD for all test rows)
    for k in (1, 2, 3):
        d[f"sin{k}"] = np.sin(2 * np.pi * k * d.tmin / 1440)
        d[f"cos{k}"] = np.cos(2 * np.pi * k * d.tmin / 1440)
    # Temperature: median impute + missing flag
    d["Temp_missing"] = d.Temperature.isna().astype(float)
    d["Temperature"]  = d.Temperature.fillna(train.Temperature.median())
    d["NumberofLanes"] = d.NumberofLanes.astype(float)
    # Categoricals: explicit "Missing" level
    for c in CAT_COLS:
        d[c] = d[c].fillna("Missing").astype("category")
    # Geohash as native LGBM categorical (avoids all-day-mean bias of target encoding)
    d["gh6"] = d.geohash.astype("category")
    return d

trf = build_features(train)
tef = build_features(test)
print("Features built. lat range:", round(trf.lat.min(), 3), "–", round(trf.lat.max(), 3))

In [ ]:
# ── Clean lag feature: day-48 demand at same (geohash, tmin) ──
# Day-48 rows in the training set are set to NaN (no self-reference).
# 88.9% of test rows have this feature; the remaining 11.1% are handled by LGBM's
# built-in NaN routing (goes to the best split direction).
d48_lkp = train[train.day == 48].set_index(["geohash", "tmin"])["demand"]

def add_lag(df):
    lag = pd.Series(
        df.set_index(["geohash", "tmin"]).index.map(d48_lkp),
        index=df.index
    ).astype(float)
    if "day" in df.columns:
        lag[df["day"] == 48] = np.nan   # no self-reference for day-48 rows
    df["d48_demand"] = lag

add_lag(trf)
add_lag(tef)

print(f"Lag coverage — day49 train: {trf[trf.day==49]['d48_demand'].notna().mean():.3f}")
print(f"Lag coverage — test:        {tef['d48_demand'].notna().mean():.3f}")

In [ ]:
# ── Feature list ──
HARM     = [f"{p}{k}" for k in (1, 2, 3) for p in ("sin", "cos")]
NUM      = ["lat", "lon", "tmin", "hour", "NumberofLanes", "Temperature", "Temp_missing"] + HARM
ALL_CATS = CAT_COLS + ["gh6"]
FEATS    = NUM + ALL_CATS + ["d48_demand"]

y = trf["demand"].values

print(f"Total features: {len(FEATS)}")
print(f"  numeric:      {len(NUM) + 1}  (+ d48_demand)")
print(f"  categorical:  {len(ALL_CATS)}")

In [ ]:
# ── Out-of-fold validation (3-fold) ──
LP = dict(
    n_estimators=1000, learning_rate=0.03, num_leaves=127,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    min_child_samples=10, n_jobs=-1, verbose=-1
)

kf  = KFold(3, shuffle=True, random_state=42)
oof = np.zeros(len(trf))

for tri, vai in kf.split(trf):
    m = lgb.LGBMRegressor(**LP, random_state=42)
    m.fit(trf.iloc[tri][FEATS], y[tri], categorical_feature=ALL_CATS)
    oof[vai] = m.predict(trf.iloc[vai][FEATS])

print(f"3-fold OOF R²: {r2_score(y, oof):.4f}")
print(f"OOF score:     {100 * r2_score(y, oof):.2f}  (gap to LB is ~4–5 pp due to day-49 shift)")

In [ ]:
# ── 3-seed bagged full-train predictions ──
# Averaging over 3 independent seeds reduces variance.
# Empirically, 5+ seeds degraded LB score; 3 was optimal.
SEEDS = [42, 123, 2024]
test_preds = []

for s in SEEDS:
    m = lgb.LGBMRegressor(
        **LP,
        random_state=s,
        bagging_seed=s,
        feature_fraction_seed=s
    )
    m.fit(trf[FEATS], y, categorical_feature=ALL_CATS)
    pred = np.clip(m.predict(tef[FEATS]), 0.0, 1.0)
    test_preds.append(pred)
    print(f"  seed {s}: mean={pred.mean():.4f}  min={pred.min():.4f}  max={pred.max():.4f}")

final_pred = np.mean(test_preds, axis=0)

# ── Save submission ──
final_sub = pd.DataFrame({"Index": tef["Index"].values, "demand": final_pred})
final_sub.to_csv("submission_final.csv", index=False)

print(f"\nsubmission_final.csv saved")
print(f"shape={final_sub.shape}  "
      f"mean={final_sub.demand.mean():.4f}  "
      f"range=[{final_sub.demand.min():.4f}, {final_sub.demand.max():.4f}]")
print(f"\nACTUAL LB score: 91.50  (this file = submission_v9_bag.csv)")
final_sub.head()